| Event                                  |       Date |
|----------------------------------------|-----------:|
| Start of first lockdown                | 2020-03-24 |
| Easing phase one                       | 2020-05-29 |
| Easing phase two                       | 2020-06-19 |
| Easing phase three                     | 2020-07-10 |
| Five-tier restriction                  | 2020-11-02 |
| Start of second lockdown — mainland    | 2021-01-05 |
| Start of vaccination                   | 2021-01-25 |
| Start of Euro 2020 football tournament | 2021-06-11 |
| End of Euro 2020 football tournament   | 2021-07-11 |
| Start of COP26                         | 2021-10-31 |
| End of COP26                           | 2021-11-11 |

In [9]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from utils import data

In [2]:
long_pd = data.load_individual_features(format="long")
long_pd = long_pd.to_pandas()

In [3]:
long_pd["seq_window"] = (long_pd["sequence_id"].astype(str)
                     + "_"
                     + long_pd["window_id"].astype(str))

long_pd["seq_window"] = long_pd["seq_window"].astype("category")

In [4]:
waves = [
    'WV1_B.1.177_C108360',
    'WV2_B.1.1.7_C574152',
    'WV3_AY.4_C983568',
     'WV4_BA.2_C479360',
    'WV5_BA.2_C85080',
    'WV6_BA.5.2_C23016'
]

sample_df = long_pd[long_pd["wave"].isin([waves[0], waves[2], waves[-1]])].copy()

sample_df["seq_window"] = sample_df["seq_window"].cat.remove_unused_categories()
sample_df["wave"] = sample_df["wave"].cat.remove_unused_categories()

In [7]:
formula1 = """
in_non_singleton ~ C(dz_simd_quintile, Treatment(3)) + C(age_group, Treatment('40–59')) + C(wave, Treatment('WV3_AY.4_C983568')) + is_female + resolution
"""

model1 = smf.gee(
    formula1,
    data=sample_df,
    groups="seq_window",
    offset=sample_df["log_seq_prop"],
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

result1 = model1.fit()
print(result1.summary())

                               GEE Regression Results                              
Dep. Variable:            in_non_singleton   No. Observations:              2293464
Model:                                 GEE   No. clusters:                   286683
Method:                        Generalized   Min. cluster size:                   8
                      Estimating Equations   Max. cluster size:                   8
Family:                           Binomial   Mean cluster size:                 8.0
Dependence structure:         Exchangeable   Num. iterations:                    22
Date:                     Sat, 25 Apr 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         00:36:01
                                                                    coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------

In [11]:
formula2 = """
in_non_singleton ~ C(dz_simd_quintile, Treatment(3)) * C(age_group, Treatment('40–59')) + C(wave, Treatment('WV3_AY.4_C983568')) + is_female + resolution
"""

model2 = smf.gee(
    formula2,
    data=sample_df,
    groups="seq_window",
    offset=sample_df["log_seq_prop"],
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

result2 = model2.fit()
print(result2.summary())

                               GEE Regression Results                              
Dep. Variable:            in_non_singleton   No. Observations:              2293464
Model:                                 GEE   No. clusters:                   286683
Method:                        Generalized   Min. cluster size:                   8
                      Estimating Equations   Max. cluster size:                   8
Family:                           Binomial   Mean cluster size:                 8.0
Dependence structure:         Exchangeable   Num. iterations:                    22
Date:                     Sat, 25 Apr 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         00:48:19
                                                                                         coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------

In [13]:
formula3 = """
in_non_singleton ~ C(dz_simd_quintile, Treatment(3)) + C(age_group, Treatment('40–59')) * C(wave, Treatment('WV3_AY.4_C983568')) + is_female + resolution
"""

model3 = smf.gee(
    formula3,
    data=sample_df,
    groups="seq_window",
    offset=sample_df["log_seq_prop"],
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

result3 = model3.fit()
print(result3.summary())

                               GEE Regression Results                              
Dep. Variable:            in_non_singleton   No. Observations:              2293464
Model:                                 GEE   No. clusters:                   286683
Method:                        Generalized   Min. cluster size:                   8
                      Estimating Equations   Max. cluster size:                   8
Family:                           Binomial   Mean cluster size:                 8.0
Dependence structure:         Exchangeable   Num. iterations:                    28
Date:                     Sat, 25 Apr 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         01:07:06
                                                                                                                coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------

In [14]:
formula4 = """
in_non_singleton ~ C(dz_simd_quintile, Treatment(3)) * C(age_group, Treatment('40–59')) * C(wave, Treatment('WV3_AY.4_C983568')) + is_female + resolution
"""

model4 = smf.gee(
    formula4,
    data=sample_df,
    groups="seq_window",
    offset=sample_df["log_seq_prop"],
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

result4 = model4.fit()
print(result4.summary())

                               GEE Regression Results                              
Dep. Variable:            in_non_singleton   No. Observations:              2293464
Model:                                 GEE   No. clusters:                   286683
Method:                        Generalized   Min. cluster size:                   8
                      Estimating Equations   Max. cluster size:                   8
Family:                           Binomial   Mean cluster size:                 8.0
Dependence structure:         Exchangeable   Num. iterations:                    53
Date:                     Sat, 25 Apr 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         01:18:40
                                                                                                                                                       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------

In [15]:
models = {
    "model1": result1,
    "model2": result2,
    "model3": result3,
    "model4": result4,
}

def extract(name, res):
    qic, qicu = res.qic(scale=1)
    ci = res.conf_int()
    return pd.DataFrame({
        "model":     name,
        "aic":       res.aic,
        "qic":       qic,
        "qicu":      qicu,
        "rho":       res.cov_struct.dep_params,
        "scale":     res.scale,
        "nobs":      res.nobs,
        "converged": res.converged,
        "term":      res.params.index,
        "coef":      res.params.values,
        "se":        res.bse.values,
        "z":         res.tvalues.values,
        "pvalue":    res.pvalues.values,
        "ci_lo":     ci.iloc[:, 0].values,
        "ci_hi":     ci.iloc[:, 1].values,
    })

results = pd.concat([extract(name, res) for name, res in models.items()], ignore_index=True)
results.to_csv("regression_analysis_results_adj_resolution.csv", index=False)